In [8]:
import numpy as np
import scipy as sp
import scipy.linalg as la

# -----------------------------
# DOF mapping helpers
# -----------------------------
DOF = {"u": 0, "v": 1, "w": 2, "thx": 3, "thy": 4, "thz": 5}

def gdof(node: int, dof_name: str, ndpn: int = 6) -> int:
    """
    Map (1-based node, dof name) -> 0-based global DOF index.
    DOF order per node: [u, v, w, thx, thy, thz]
    """
    return (node - 1) * ndpn + DOF[dof_name]



def timoshenko_beam_3d_stiffness(E, G, A, L, Iy, Iz, J, phi_y, phi_z):
    """
    12x12 local stiffness matrix matching the form in the provided image.

    DOF order:
    [u1, v1, w1, thx1, thy1, thz1, u2, v2, w2, thx2, thy2, thz2]
    """

    K = np.zeros((12, 12), dtype=float)

    # --- Axial ---
    k_ax = E * A / L
    K[0, 0] =  k_ax
    K[0, 6] = -k_ax
    K[6, 0] = -k_ax
    K[6, 6] =  k_ax

    # --- Torsion ---
    k_tor = G * J / L
    K[3, 3]   =  k_tor
    K[3, 9]   = -k_tor
    K[9, 3]   = -k_tor
    K[9, 9]   =  k_tor

    # --- Bending associated with v and thz (uses Iz, phi_y) ---
    kvv = 12 * E * Iz / (L**3 * (1 + phi_y))
    kvt =  6 * E * Iz / (L**2 * (1 + phi_y))
    ktt = (4 + phi_y) * E * Iz / (L * (1 + phi_y))
    ktt2 = (2 - phi_y) * E * Iz / (L * (1 + phi_y))

    # indices: v1=1, thz1=5, v2=7, thz2=11
    v1, thz1, v2, thz2 = 1, 5, 7, 11

    K[v1, v1]     =  kvv
    K[v1, thz1]   =  kvt
    K[v1, v2]     = -kvv
    K[v1, thz2]   =  kvt

    K[thz1, v1]   =  kvt
    K[thz1, thz1] =  ktt
    K[thz1, v2]   = -kvt
    K[thz1, thz2] =  ktt2

    K[v2, v1]     = -kvv
    K[v2, thz1]   = -kvt
    K[v2, v2]     =  kvv
    K[v2, thz2]   = -kvt

    K[thz2, v1]   =  kvt
    K[thz2, thz1] =  ktt2
    K[thz2, v2]   = -kvt
    K[thz2, thz2] =  ktt

    # --- Bending associated with w and thy (uses Iy, phi_z) ---
    kww = 12 * E * Iy / (L**3 * (1 + phi_z))
    kwt =  6 * E * Iy / (L**2 * (1 + phi_z))
    kyy = (4 + phi_z) * E * Iy / (L * (1 + phi_z))
    kyy2 = (2 - phi_z) * E * Iy / (L * (1 + phi_z))

    # indices: w1=2, thy1=4, w2=8, thy2=10
    w1, thy1, w2, thy2 = 2, 4, 8, 10

    K[w1, w1]     =  kww
    K[w1, thy1]   = -kwt
    K[w1, w2]     = -kww
    K[w1, thy2]   = -kwt

    K[thy1, w1]   = -kwt
    K[thy1, thy1] =  kyy
    K[thy1, w2]   =  kwt
    K[thy1, thy2] =  kyy2

    K[w2, w1]     = -kww
    K[w2, thy1]   =  kwt
    K[w2, w2]     =  kww
    K[w2, thy2]   =  kwt

    K[thy2, w1]   = -kwt
    K[thy2, thy1] =  kyy2
    K[thy2, w2]   =  kwt
    K[thy2, thy2] =  kyy

    return K

def assemble_1d_chain_4nodes(K_e_list, conn, ndof_per_node=6):
    """
    Assemble global stiffness for a 1D chain.

    K_e_list: list of 12x12 element stiffness matrices (one per element)
    conn:     list of (node_i, node_j) connectivity using 1-based node numbers
              e.g. [(1,2),(2,3),(3,4)]
    """
    n_nodes = 4
    ndof = n_nodes * ndof_per_node
    K_global = np.zeros((ndof, ndof))

    for e, (ni, nj) in enumerate(conn):
        Ke = K_e_list[e]

        # convert node numbers (1-based) to global dof indices (0-based)
        dofs_i = list(range((ni-1)*ndof_per_node, ni*ndof_per_node))
        dofs_j = list(range((nj-1)*ndof_per_node, nj*ndof_per_node))
        edofs = dofs_i + dofs_j  # 12 dofs for this element

        # assemble: add Ke into K_global at the edofs positions
        for a in range(12):
            A = edofs[a]
            for b in range(12):
                B = edofs[b]
                K_global[A, B] += Ke[a, b]

    return K_global

# -----------------------------
# Reduce K, M with Dirichlet BCs (eliminate constrained dofs)
# -----------------------------
def reduce_matrices(K, M, constrained_dofs):
    n = K.shape[0]
    constrained_dofs = np.array(sorted(set(constrained_dofs)), dtype=int)
    all_dofs = np.arange(n, dtype=int)
    free_dofs = np.setdiff1d(all_dofs, constrained_dofs)

    Kff = K[np.ix_(free_dofs, free_dofs)]
    Mff = M[np.ix_(free_dofs, free_dofs)]
    return Kff, Mff, free_dofs, constrained_dofs

def expand_modes(evecs_free, free_dofs, ndof_total):
    modes = np.zeros((ndof_total, evecs_free.shape[1]), dtype=float)
    modes[free_dofs, :] = evecs_free
    return modes

def mass_normalize_modes(modes, M):
    modes_n = modes.copy()
    for i in range(modes.shape[1]):
        mi = modes[:, i].T @ (M @ modes[:, i])
        modes_n[:, i] /= np.sqrt(mi)
    return modes_n


nd=4
n_elms=nd-1
rho=9000
A=0.5*0.5
L=5
l_elms=L/n_elms
m_e=rho*A*l_elms

m = np.full(nd, m_e) 
m[0] = m_e / 2
m[-1] = m_e / 2
zero = np.zeros((3,3))
blocks = []

for k in range(len(m)):
    blocks.append(zero)              # zero block after it
    blocks.append(m[k] * np.eye(3))  # mass block
   

M = sp.linalg.block_diag(*blocks)


# connectivity for 3 elements
conn = [(1,2), (2,3), (3,4)]

# build each element stiffness (can differ per element if properties differ)
K_e_list = []
for e in range(n_elms):
    Ke = timoshenko_beam_3d_stiffness(
    E=210e9, G=80e9, A=0.01, L=2.0,
    Iy=1e-6, Iz=2e-6, J=5e-7,
    phi_y=0.2, phi_z=0.3
    )
    K_e_list.append(Ke)

K = assemble_1d_chain_4nodes(K_e_list, conn, ndof_per_node=6)



constrained = [gdof(1, d) for d in ["u", "v", "w", "thx", "thy", "thz"]]

Kff, Mff, free, cons = reduce_matrices(K, M, constrained)

    # ============================================================
    # SOLVE GENERALIZED EIGENPROBLEM: Kff phi = lambda Mff phi
    # ============================================================
nmodes = 6  # number of modes you want
evals, evecs = la.eigh(Kff, Mff)

    # Convert to frequencies
evals = np.maximum(evals, 0.0)
omega = np.sqrt(evals)          # rad/s
freq  = omega / (2*np.pi)       # Hz

    # Expand eigenvectors back to full DOF space (zeros at constrained dofs)
modes_full = expand_modes(evecs, free, K.shape[0])

    # Mass-normalize the full modes (optional but common)
modes_full = mass_normalize_modes(modes_full, M)

print("Eigenvalues (lambda = omega^2):")
print(evals)
print("\nNatural frequencies (Hz):")
print(freq)

LinAlgError: The leading minor of order 1 of B is not positive definite. The factorization of B could not be completed and no eigenvalues or eigenvectors were computed.